# UK Biobank research content, 2013–2025

This is the single active content-analysis notebook. It consolidates the BERTopic and category-flow notebooks, preserved under `_archived/`.

The main figure places **A, FOR Level 4 composition** and **B, RCDC composition** side by side, with **C, thematic topic waves** below. Two supplements show annual category detail and classification coverage/breadth; a third shows topic-model robustness when its verified diagnostics are available. Figures use the shared project palette, Helvetica, bold left-aligned uppercase titles, and 500-DPI PNG plus PDF exports.

All analysis is restricted to **1 January 2013–31 December 2025 inclusive**. Topic modelling and figure generation are independent: completed topic assignments are checked before any modelling libraries are imported. Existing results are reused, while missing results trigger the preserved five-configuration, five-seed robustness workflow. Invalid or unverified results stop without a silent refit.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
P.bootstrap()

## 1. Reuse or compute topic results

Leave the defaults for a complete run. Set `RUN_TOPIC_MODELLING = False` to inspect the classification figures without fitting if results are absent. Set `FORCE_TOPIC_REFIT = True` only for an intentional refit.

The cache checks `output/bertopic/` and registered legacy CSV locations. Results must contain stable publication IDs, assignments, and a matching analysis-window provenance sidecar. Cached embeddings alone are not completed topic results. Embeddings and each successful parameter/seed fit are reused independently if a run is interrupted. The original SPECTER cache can be reused only after reproducing its ordered-corpus hash and matching the current documents; the clustering itself uses only the current analysis window.

In [ ]:
RUN_TOPIC_MODELLING = os.environ.get("UKB_RUN_TOPIC_MODELLING", "1") != "0"
FORCE_TOPIC_REFIT = False

from utils import data_analysis_02_content_topics as T
TOPIC_RESULTS = T.ensure_topic_results(
    train_if_missing=RUN_TOPIC_MODELLING,
    force=FORCE_TOPIC_REFIT,
)

## 2. Prepare the shared figure data

FOR composition counts a publication once in each assigned Level 4 field, then divides by all paper–field assignments in that year. RCDC divides one unit per paper across its distinct tags. BERTopic supplies one topic per eligible publication after the existing outlier-reassignment procedure. Missing classifications remain outside composition denominators and are quantified in the coverage supplement.

The main figure uses leading categories selected by total in-window weight; the full distributions remain in the source tables. The topic-wave thickness is the annual share of **all non-outlier topic-assigned publications**, without renormalising the displayed topics to 100%. The vertical baseline is a display convention. Very small early cohorts are explicitly reported.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from utils.shared_style import PNG_DPI, load_style, savefig
from utils.shared_analysis_window import ANALYSIS_START_YEAR, ANALYSIS_END_YEAR
from utils import data_analysis_02_content_panels as C
from utils import data_analysis_02_content_exports as E

STYLE = load_style("02_content")
FLOW_MIN, FLOW_MAX = ANALYSIS_START_YEAR, ANALYSIS_END_YEAR
FLOW_YEARS = list(range(FLOW_MIN, FLOW_MAX + 1))
ARTIFACTS = P.ArtifactRegistry(P.TABLE_CONTENT)
D = C.build_panel_data(topic_results=TOPIC_RESULTS)
display(C.band_summary(D))

## 3. Main figure: research composition and thematic waves

Panels are drawn directly from the shared aggregates. Rerunning this cell changes the figure without fitting a topic model. Full model labels and displayed abbreviations are retained in `content_topic_labels.csv`.

In [ ]:
fig = C.figure_main(D, save=False)
main_stem = "02_01_figure_01_content_composition"
if not D["topics"]["available"]:
    main_stem += "_incomplete"
    print("[WARN] Topic results are absent; this is an incomplete preview.")
ARTIFACTS.record_figures(savefig(fig, main_stem, style=STYLE))
display(fig)
plt.close(fig)

## 4. Supplementary figure: annual category detail

Annual shares of the twelve leading fields and RCDC tags, with percentage annotations and cell borders. The shared cool–cream–red colormap spans the full observed range within each panel. These are whole-vocabulary shares, so the twelve displayed rows need not sum to 100%.

In [ ]:
fig = C.figure_si_category_changes(D, save=False)
ARTIFACTS.record_figures(savefig(
    fig, "02_02_supplementary_figure_01_category_composition", style=STYLE))
display(fig)
plt.close(fig)

## 5. Supplementary figure: coverage and breadth

Coverage uses all papers in each year as denominator. Diversity is the exponential of Shannon entropy. Annual and cumulative category counts describe each vocabulary separately; different vocabularies have different granularity, and counts are not equivalent units. These descriptive measures also depend on corpus size.

In [ ]:
fig = C.figure_si_breadth_coverage(D, save=False)
ARTIFACTS.record_figures(savefig(
    fig, "02_03_supplementary_figure_02_coverage_and_breadth", style=STYLE))
display(fig)
plt.close(fig)

## 6. Supplementary figure: topic-model robustness

When the completed run supplies verified diagnostics, report variation across the five parameter configurations and five seeds, assignment stability, and final HDBSCAN cluster persistence. Diagnostic plots read saved tables and never fit a model. The representative seed is the assignment medoid under adjusted Rand agreement; the primary parameter score combines coherence, diversity, outlier rate, topic-count penalty and between-seed stability. Persistence is an additional diagnostic.

In [ ]:
diagnostics = C.load_topic_diagnostics(D)
if diagnostics is not None:
    fig = C.figure_si_topic_robustness(D, save=False)
    ARTIFACTS.record_figures(savefig(
        fig, "02_04_supplementary_figure_03_topic_robustness", style=STYLE))
    display(fig)
    plt.close(fig)
else:
    print("[SKIP] Verified topic-model robustness diagnostics are unavailable.")

## 7. Source tables, captions and manifest

Exports include full annual category distributions, denominators, category-share changes, displayed-band coverage, topic labels and analysis parameters. The caption and methods files document weighting, selection and the treatment of missing data. Figures are PNG/PDF; tables are CSV. JSON sidecars are used only for model-cache provenance.

The consolidated workflow removes the old hard-coded historical count comparisons and duplicate standalone figures. Earlier notebook implementations remain available in `_archived/` for reference.

In [ ]:
tables = E.export_content_tables(D, ARTIFACTS)
manifest, manifest_path = ARTIFACTS.save_manifest("02_content_manifest.csv")
print(f"Exported {len(ARTIFACTS.figure_paths)} figure files and "
      f"{len(ARTIFACTS.table_paths)} tables/captions; PNG resolution {PNG_DPI} DPI.")
print("Manifest:", P.raw_path(manifest_path))